# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# AE index

In [ ]:
import pyspedas as psp

time_range = ['20220901/20:00:00', '20220902/00:00:00']

psp.projects.omni.data(trange=time_range)

da_ae_index = psp.get_data('AE_INDEX', xarray=True)
da_au_index = psp.get_data('AU_INDEX', xarray=True)
da_al_index = psp.get_data('AL_INDEX', xarray=True)

print(da_ae_index)
print(da_au_index)
print(da_al_index)

# $\mathbf{B}$ (THEMIS-A, GSM)

In [ ]:
import pyspedas as psp
import xarray as xr

psp.projects.themis.fgm(trange=time_range, probe='a', level='l2', get_support_data=True)    # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec

da_Bspin_data_gsm   = psp.get_data('tha_fgs_gsm', xarray=True)
da_Bspin_data_gsm   = da_Bspin_data_gsm.sortby('time').sel(time=slice(time_range[0], time_range[1]))

print(da_Bspin_data_gsm)

In [ ]:
import numpy as np

background_time_sec = 100 #[sec]

time_width_B_spin = (da_Bspin_data_gsm.time[10] - da_Bspin_data_gsm.time[9]) / np.timedelta64(1, 's')
da_Bspin_0_data_gsm = da_Bspin_data_gsm.rolling(time=int(background_time_sec/time_width_B_spin), center=True).mean('time').dropna(how='all', dim='time')

In [ ]:
da_Bspin_0_x = da_Bspin_0_data_gsm[:, 0]
da_Bspin_0_y = da_Bspin_0_data_gsm[:, 1]
da_Bspin_0_z = da_Bspin_0_data_gsm[:, 2]

import numpy as np
import xarray as xr

theta_GSM_0 = np.rad2deg(np.arctan(da_Bspin_0_z / np.sqrt(da_Bspin_0_x**2 + da_Bspin_0_y**2)))

da_theta_GSM_0 = xr.DataArray(
    data=theta_GSM_0,
    dims=['time'],
    coords={'time': da_Bspin_0_data_gsm.time},
    name='theta_GSM_0_deg'
)

print(da_theta_GSM_0)

In [ ]:
da_Bspin_x = da_Bspin_data_gsm[:, 0]
da_Bspin_y = da_Bspin_data_gsm[:, 1]
da_Bspin_z = da_Bspin_data_gsm[:, 2]

import numpy as np
import xarray as xr

theta_GSM = np.rad2deg(np.arctan(da_Bspin_z / np.sqrt(da_Bspin_x**2 + da_Bspin_y**2)))

da_theta_GSM = xr.DataArray(
    data=theta_GSM,
    dims=['time'],
    coords={'time': da_Bspin_data_gsm.time},
    name='theta_GSM_deg'
)

print(da_theta_GSM)

# plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime

mpl.rcParams['font.size'] = 20

time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_ae_index_analysis    = da_ae_index.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_au_index_analysis    = da_au_index.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_al_index_analysis    = da_al_index.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))

da_Bspin_0_x_analysis   = da_Bspin_0_x.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_Bspin_0_y_analysis   = da_Bspin_0_y.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_Bspin_0_z_analysis   = da_Bspin_0_z.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))

da_theta_GSM_0_analysis   = da_theta_GSM_0.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))

da_Bspin_x_analysis   = da_Bspin_x.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_Bspin_y_analysis   = da_Bspin_y.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))
da_Bspin_z_analysis   = da_Bspin_z.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))

da_theta_GSM_analysis   = da_theta_GSM.sel(time=slice(time_range_analysis[0], time_range_analysis[1]))


fig = plt.figure(figsize=(15, 18))

gs = fig.add_gridspec(5, 1, height_ratios=[3, 1, 1, 1, 2], hspace=0.05)
ax_1 = fig.add_subplot(gs[0, 0])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
ax_5 = fig.add_subplot(gs[4, 0], sharex=ax_1)

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)

ax_1.plot(da_al_index_analysis.time, da_al_index_analysis.data, lw=1, c='r', label='AL')
ax_1.plot(da_au_index_analysis.time, da_au_index_analysis.data, lw=1, c='b', label='AU')
ax_1.plot(da_ae_index_analysis.time, da_ae_index_analysis.data, lw=1, c='purple', label='AE')
ax_1.set_ylabel(r'AL, AU, AE indices'   + '\n' + '[nT]')
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_1.legend(ncol=3, fontsize=15)

ax_2.plot(da_Bspin_0_x_analysis.time, da_Bspin_0_x_analysis.data, lw=2, c='k')
ax_2.plot(da_Bspin_x_analysis.time, da_Bspin_x_analysis.data, lw=2, c='gray', alpha=0.5)
ax_2.set_ylabel(r'$B_{x}$ (GSM)'   + '\n' + '[nT]')
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)

ax_3.plot(da_Bspin_0_y_analysis.time, da_Bspin_0_y_analysis.data, lw=2, c='k')
ax_3.plot(da_Bspin_y_analysis.time, da_Bspin_y_analysis.data, lw=2, c='gray', alpha=0.5)
ax_3.set_ylabel(r'$B_{y}$ (GSM)'   + '\n' + '[nT]')
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)

ax_4.plot(da_Bspin_0_z_analysis.time, da_Bspin_0_z_analysis.data, lw=2, c='k')
ax_4.plot(da_Bspin_z_analysis.time, da_Bspin_z_analysis.data, lw=2, c='gray', alpha=0.5)
ax_4.set_ylabel(r'$B_{z}$ (GSM)'   + '\n' + '[nT]')
ax_4.minorticks_on()
ax_4.grid(which='both', alpha=0.5)

ax_5.plot(da_theta_GSM_0_analysis.time, da_theta_GSM_0_analysis.data, lw=2, c='k')
ax_5.plot(da_theta_GSM_analysis.time, da_theta_GSM_analysis.data, lw=2, c='gray', alpha=0.5)
ax_5.set_ylabel(r'$\theta_{\mathrm{GSM}}$'   + '\n' + '[deg]')
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)
ax_5.set_ylim(70, 90)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

def add_panel_label(ax, label, x=-0.07, y=0.90):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

add_panel_label(ax_1, '(a)')
add_panel_label(ax_2, '(b)')
add_panel_label(ax_3, '(c)')
add_panel_label(ax_4, '(d)')
add_panel_label(ax_5, '(e)')

fig.tight_layout()

fig.savefig(r"/mnt/j/KAW_observation/AE_index_B_GSM.pdf", bbox_inches='tight')
fig.savefig(r"/mnt/j/KAW_observation/AE_index_B_GSM.png", bbox_inches='tight')